In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
import matplotlib.pyplot as plt

from backtest import load_data, run_backtest
from strategies.sma_cross import SmaCross
from strategies.rsi_reversion import RsiReversion
from strategies.bollinger_reversion import BollingerReversion
from strategies.donchian_breakout import DonchianBreakout
from strategies.ts_momentum import TimeSeriesMomentum
from strategies.macd_cross import MacdCross

In [ ]:
strategies = {
    "SmaCross":           SmaCross,
    "RsiReversion":       RsiReversion,
    "BollingerReversion": BollingerReversion,
    "DonchianBreakout":   DonchianBreakout,
    "TimeSeriesMomentum": TimeSeriesMomentum,
    "MacdCross":          MacdCross,
}

METRICS = [
    "Return [%]",
    "Buy & Hold Return [%]",
    "Sharpe Ratio",
    "Max. Drawdown [%]",
    "Win Rate [%]",
    "# Trades",
]

results = {}   # tf -> {strategy -> stats}

for tf in ["1h", "4h"]:
    df = load_data("BTC/USDT", tf)
    tf_stats = {}
    for name, strat in strategies.items():
        _, stats = run_backtest(df, strat)
        tf_stats[name] = stats
    results[tf] = tf_stats
    
    rows = {}
    for metric in METRICS:
        rows[metric] = {name: round(stats[metric], 2) if not isinstance(stats[metric], int) else stats[metric]
                        for name, stats in tf_stats.items()}
    
    comparison = pd.DataFrame(rows).T
    print(f"\n=== {tf} 비교표 (5년: 2021-06 ~ 2026-06, 2022 폭락 포함) ===")
    display(comparison)

In [ ]:
# B&H 최대낙폭 계산 (4h 종가 기준)
df_4h = load_data("BTC/USDT", "4h")
c = df_4h["Close"]
dd = c / c.cummax() - 1
bnh_mdd = dd.min()
print(f"B&H 최대낙폭 (4h close): {bnh_mdd:.4f} = {bnh_mdd*100:.2f}%")
print(f"기간: {c.index[0]} ~ {c.index[-1]}")
print(f"시작가: ${c.iloc[0]:,.0f}")
print(f"최저가: ${c.min():,.0f}")
print(f"종가: ${c.iloc[-1]:,.0f}")
bnh_5yr_return = (c.iloc[-1] / c.iloc[0] - 1) * 100
print(f"B&H 5년 수익: {bnh_5yr_return:.2f}%")

In [ ]:
# 4h 6전략 자본곡선 오버레이
fig, ax = plt.subplots(figsize=(14, 6))

for name, stats in results["4h"].items():
    equity = stats["_equity_curve"]["Equity"]
    ax.plot(equity.index, equity.values, label=name)

ax.set_title("6전략 자본곡선 비교 (BTC/USDT 4h)\n5년 (2021-06~2026-06, 2022 폭락 포함)")
ax.set_xlabel("Date")
ax.set_ylabel("Equity (USD)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()